# 01c — So sánh protocol 4 task và 5 task cho AutoCheckout-CL

Ngày tạo: 07/10/2026. Mục đích: đọc config thực tế, so sánh cách chia SKU/nhóm hàng và ranh giới task; đưa ra khuyến nghị phù hợp với bài toán. Đây là phân tích protocol, không train model, không tạo split và không tự thay quyết định của người dùng.

Nguồn repo: `data/archive/tasks/group/task_info.json`, `configs/tasks_100-4x25_seed0.json`. Nguồn nghiên cứu: [On Class Orderings for Incremental Learning](https://arxiv.org/abs/2007.02145), cho thấy class order có thể thay đổi hiệu năng và thứ hạng phương pháp trên benchmark classification. Không chuyển kết quả đó thành bằng chứng protocol nào đạt cAcc cao hơn trong dự án này.


**Cập nhật sau lần chạy này:** người dùng đã chốt protocol **5 task 100+4×25**. Split/task JSON ảnh thật đã nghiệm thu trong [notebook 02](02_split_and_acceptance.ipynb); xem [báo cáo release](../../docs/data_preprocessing/reports/split-100-4x25-2026-10-07.md). Notebook này giữ output của lần audit/phân tích trước quyết định; nhãn synth được review riêng trong [notebook 03](03_annotation_review.ipynb).


In [1]:
from pathlib import Path
import json,subprocess,sys,uuid,hashlib
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo
TZ=ZoneInfo('Asia/Bangkok')
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pdp/main.py').exists())
started_at=datetime.now(TZ).isoformat(timespec='seconds')
run={'run_id':str(uuid.uuid4()),'purpose':'compare actual 4-task grouped and 5-task mixed SKU protocols','started_at':started_at,
     'finished_at':None,'timezone':'Asia/Bangkok','python':sys.version.split()[0],
     'git_commit':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),
     'config_sha256':{rel:hashlib.sha256((REPO/rel).read_bytes()).hexdigest() for rel in ['data/archive/tasks/group/task_info.json','configs/tasks_100-4x25_seed0.json']},
     'training_executed':False,'mapping_changed':False,'protocol_decision':'pending user choice'}
four=json.loads((REPO/'data/archive/tasks/group/task_info.json').read_text())
five=json.loads((REPO/'configs/tasks_100-4x25_seed0.json').read_text())
print(json.dumps(run,ensure_ascii=False,indent=2))


{
  "run_id": "d77b0aac-e711-4351-ae8b-348412cba16e",
  "purpose": "compare actual 4-task grouped and 5-task mixed SKU protocols",
  "started_at": "2026-10-07T15:23:52+07:00",
  "finished_at": null,
  "timezone": "Asia/Bangkok",
  "python": "3.14.4",
  "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
  "config_sha256": {
    "data/archive/tasks/group/task_info.json": "87f276801b9e19c14176a62e1a531ed2b13bdb56b98bb3c30b5d50499accd7a5",
    "configs/tasks_100-4x25_seed0.json": "be12b57f4c28b17597794ad526012a5ba3a72b98dd48739e2c31606f7240cf9a"
  },
  "training_executed": false,
  "mapping_changed": false,
  "protocol_decision": "pending user choice"
}


## Số liệu từ config

`group_4`: SKU cùng nhóm hàng vào cùng task; các nhóm hàng không lặp qua các task. `mixed_5`: mỗi task có SKU của cả 17 nhóm hàng. Tỷ lệ old/new dưới đây tính theo số lớp, không phải số vật hoặc pseudo-label trên ảnh.


In [2]:
rows=[];catalogs={};group_sets={}
for protocol,tasks in [('group_4',four['tasks']),('mixed_5',[t for t in five['tasks'] if not t.get('reserved')])]:
    seen=0;class_ids=[];groups_per_task=[]
    for task in tasks:
        if protocol=='group_4':
            ids=task['new_classes'];groups=dict(task['groups']);task_id=task['task']
        else:
            ids=[c['rpc_category_id'] for c in task['classes']]
            groups=dict(Counter(c['supercategory'] for c in task['classes']));task_id=task['task_id']
        class_ids.extend(ids);groups_per_task.append(set(groups))
        rows.append({'protocol':protocol,'task':task_id,'old_classes':seen,'new_classes':len(ids),
                     'seen_after':seen+len(ids),'supercategory_count':len(groups),
                     'old_new_class_ratio':round(seen/len(ids),3),'groups':groups})
        seen+=len(ids)
    assert len(class_ids)==200 and len(set(class_ids))==200 and set(class_ids)==set(range(1,201))
    catalogs[protocol]=set(class_ids);group_sets[protocol]=groups_per_task
assert catalogs['group_4']==catalogs['mixed_5']
assert all(len(g)==17 for g in group_sets['mixed_5'])
assert all(not a&b for k,a in enumerate(group_sets['group_4']) for b in group_sets['group_4'][k+1:])
print('| Protocol | Task | SKU cũ | SKU mới | SKU sau đợt | Nhóm hàng | Old/new (lớp) |')
print('|---|---:|---:|---:|---:|---:|---:|')
for r in rows:
    print(f"| {r['protocol']} | {r['task']} | {r['old_classes']} | {r['new_classes']} | {r['seen_after']} | {r['supercategory_count']} | {r['old_new_class_ratio']} |")
print('PASS: cùng 200 SKU, không lặp SKU giữa các đợt; grouped có nhóm hàng rời nhau, mixed bao phủ cả 17 nhóm/task')


| Protocol | Task | SKU cũ | SKU mới | SKU sau đợt | Nhóm hàng | Old/new (lớp) |
|---|---:|---:|---:|---:|---:|---:|
| group_4 | 1 | 0 | 48 | 48 | 4 | 0.0 |
| group_4 | 2 | 48 | 68 | 116 | 6 | 0.706 |
| group_4 | 3 | 116 | 47 | 163 | 4 | 2.468 |
| group_4 | 4 | 163 | 37 | 200 | 3 | 4.405 |
| mixed_5 | 1 | 0 | 100 | 100 | 17 | 0.0 |
| mixed_5 | 2 | 100 | 25 | 125 | 17 | 4.0 |
| mixed_5 | 3 | 125 | 25 | 150 | 17 | 5.0 |
| mixed_5 | 4 | 150 | 25 | 175 | 17 | 6.0 |
| mixed_5 | 5 | 175 | 25 | 200 | 17 | 7.0 |
PASS: cùng 200 SKU, không lặp SKU giữa các đợt; grouped có nhóm hàng rời nhau, mixed bao phủ cả 17 nhóm/task


## Khuyến nghị và giới hạn

- **Protocol chính đề xuất: 5 task 100+4×25, phân tầng theo 17 nhóm hàng.** Phù hợp với kịch bản catalog ban đầu gồm nhiều loại sản phẩm rồi nhập thêm SKU theo đợt; giúp kiểm tra phân biệt SKU mới với SKU cũ cùng nhóm hàng. Đây là suy luận về mức phù hợp với mục tiêu dự án, chưa phải kết quả cAcc/mAP thực nghiệm.
- **4 task grouped:** dùng cho pilot/debug hoặc đối chứng khả năng mở rộng sang nhóm hàng chưa từng xuất hiện; thuận tiện đối chiếu notebook của nhóm. Không mặc định grouping dễ hơn: có chuyển nhóm hàng rõ và từng đợt có số SKU mới không đều.
- Task 1 của mixed bao phủ nhiều nhóm hơn, nhưng có 4 lần chuyển task thay vì 3 và tỷ lệ số lớp cũ/mới ở đợt cuối là 175/25=7; grouped là 163/37≈4.405. Không suy ra mixed luôn ít quên hơn hoặc tốn ít GPU hơn.
- Khi so ResNet với ConvNeXt, dùng **cùng protocol/mapping/split**, pretrained policy và ngân sách optimizer steps. Nếu so hai protocol, báo cáo theo cùng số lớp đã học khi có thể, cùng test cuối trên 200 SKU, cAcc/mAP/forgetting, tổng steps/thời gian và seed/order. Không lấy số task ít hơn để suy ra chi phí chính xác.
- Data synth hiện ghép theo 4 nhóm. Chọn 5 task cần kiểm tra lại phân bố SKU trên mỗi ảnh/task và remap annotations hoặc tái ghép; không chỉ đổi n_tasks. Replay-free và supervision chỉ SKU mới vẫn phải giữ.
- Các protocol RPC này là lựa chọn của dự án/nhóm; số task giống paper không đủ chứng minh tái lập benchmark paper. Thay data và backbone cần ghi riêng điều kiện thực nghiệm.

**Chưa tự chọn protocol hoặc sửa data/config.**


In [3]:
run['finished_at']=datetime.now(TZ).isoformat(timespec='seconds')
run['duration_seconds']=(datetime.fromisoformat(run['finished_at'])-datetime.fromisoformat(run['started_at'])).total_seconds()
run['status']='CONFIG_COMPARISON_COMPLETE_NO_TRAINING'
print(json.dumps({'run':run,'recommendation':'5-task mixed as main; 4-task grouped for pilot/secondary comparison',
                  'accuracy_winner':'unknown without controlled experiments','task_statistics':rows},ensure_ascii=False,indent=2))


{
  "run": {
    "run_id": "d77b0aac-e711-4351-ae8b-348412cba16e",
    "purpose": "compare actual 4-task grouped and 5-task mixed SKU protocols",
    "started_at": "2026-10-07T15:23:52+07:00",
    "finished_at": "2026-10-07T15:23:52+07:00",
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
    "config_sha256": {
      "data/archive/tasks/group/task_info.json": "87f276801b9e19c14176a62e1a531ed2b13bdb56b98bb3c30b5d50499accd7a5",
      "configs/tasks_100-4x25_seed0.json": "be12b57f4c28b17597794ad526012a5ba3a72b98dd48739e2c31606f7240cf9a"
    },
    "training_executed": false,
    "mapping_changed": false,
    "protocol_decision": "pending user choice",
    "duration_seconds": 0.0,
    "status": "CONFIG_COMPARISON_COMPLETE_NO_TRAINING"
  },
  "recommendation": "5-task mixed as main; 4-task grouped for pilot/secondary comparison",
  "accuracy_winner": "unknown without controlled experiments",
  "task_statistics": [
    {
  